# Achtergrond & Biografie van Politici

**Thema:** Opleidingsachtergrond, loopbaan en demografie.

**Kernvragen:** Wat is de gemiddelde leeftijd, opleidingsachtergrond en geografische spreiding (geboorteplaats versus woonplaats) van politici? Welke werkgevers domineren hun eerdere loopbaan?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_onderwijs` en `persoon_loopbaan`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True) #connectie met de huidige database


def query_df(sql: str, parameters=None) -> pd.DataFrame:     #functie waarmee je sql kan schrijven
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Verkenning tabellen

In [ ]:
# Verken de database
query_df("SHOW ALL TABLES")

In [ ]:
persoondf= query_df("""select *
from silver.persoon""")
persoondf

Tabellen met kolommen die relevant zijn: persoon, persoonloopbaan, onderwijs \
persoon: **geboortedatum, sterftedatum** samen een nieuwe kolom berekenen leeftijd, **geslacht**, **geboorteplaats**,**geboorteland**,**woonplaats**, **land** \
\
Datatypes: strings en datetime64\
\
\
persoonloopbaan: **Functie**,**Werkgever**, **van ruw**, **tot ruw**, **gewicht/volgorde** ,**titels** bekijken wie welke titel heeft zoals professor of mr of drs\
\
Datatypes: strings en datetime64\
\
\
persoononderwijs: **opleiding nl eng** ,**instelling**, **van tot**\
\
Datatypes: strings en datetime64


## Notities betreft tabellen
silver.persoon: hoe eerder de geboortedatum hoe meer missende waarden. Soms met geboorteplaats wel een invulling zoals maasbree maar geboorteland nederland dan weer een missing value. Geslacht altijd ingevuld. woonplaats+ land ook voor t merendeel weer zelfde patroon persoon eerder in de geschiedenis meer missingvalues vooral als ze al overleden.
sommige rijen bevatten meer dan 3 missing values\
\
silver.onderwijs: 1 persoon kan meerdere opleidingen gedaan hebben. sommige mavo havo vwo hbo/uni. komen ook cursussen voor vanuit partijen of gewoon certificaten voor management. gewicht 1 in de kolom is de laatste opleiding die gevolgd is. soms kan laatste opleiding van de partij zelf zijn.
\
persoonloopbaan: lastig te categoriseren van banen. gewicht 1 is nieuwste baan/ laatste baan.\
Steekproefsgewijs gezien dat veel huidige kamerleden hun loopbaan niet hebben opgegeven.

### QUERY's

In [ ]:
missingvaluesdf= query_df("""
    SELECT EXTRACT(decade FROM geboortedatum) AS decennium,
           COUNT(*) AS totaal,
           COUNT(geboorteland) AS met_land,
           COUNT(overlijdensdatum) AS overleden,
           COUNT(woonplaats) AS met_woonplaats
    FROM silver.persoon
    GROUP BY decennium
    ORDER BY decennium
""")
#SQL negeert missing values dus count telt dan niks

df = missingvaluesdf.copy()

# Rij met NULL-geboortedatum (decennium = NaN) apart bekijken i.p.v. stilzwijgend meenemen
print(df[df["decennium"].isna()])
df = df.dropna(subset=["decennium"])

df["decennium"] = (df["decennium"] * 10).astype(int)   # 198 -> 1980

df["land_missend_pct"] = (1 - df["met_land"] / df["totaal"]) * 100
df["woonplaats_missend_pct"] = (1 - df["met_woonplaats"] / df["totaal"]) * 100

ax = df.plot(
    x="decennium",
    y=["land_missend_pct", "woonplaats_missend_pct"],
    kind="bar",
    figsize=(10, 5),
)
ax.set_xlabel("Geboortedecennium")
ax.set_ylabel("Ontbrekende waarden (%)")
ax.set_title("Percentage ontbrekende waarden per geboortedecennium")
ax.legend(["geboorteland", "woonplaats"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
query_df("SELECT COUNT(*) AS totaal, COUNT(geslacht) AS ingevuld FROM silver.persoon")

In [ ]:
#checken waar de volgorde van gewicht op is gebassseerd
query_df("""
    SELECT persoon_id, opleiding_nl, gewicht, van_ruw
    FROM silver.persoon_onderwijs
    WHERE persoon_id = (SELECT persoon_id FROM silver.persoon_onderwijs GROUP BY persoon_id HAVING COUNT(*) > 3 LIMIT 1)
    ORDER BY gewicht
""")

In [ ]:
#checken of de politici van de huidige kamer hun loopbaan hebben opgegeven
query_df("""
    SELECT COUNT(DISTINCT p.id) AS totaal_personen,
           COUNT(DISTINCT l.persoon_id) AS met_loopbaan
    FROM silver.persoon p
    LEFT JOIN silver.persoon_loopbaan l ON p.id = l.persoon_id
    WHERE p.functie = 'Tweede Kamerlid'
""")

# Kolomnamen bekijken + waarden

In [ ]:
query_df("""Select *
   FROM silver.persoon_loopbaan

""")


In [ ]:
query_df("""Select *
   FROM silver.persoon

""")


In [ ]:
persoondataframe= query_df("""Select *
   FROM silver.persoon_loopbaan
""")
persoondataframe.isna().sum()

Na een snelle inspectie is te zien dat Voornamen, roepnamen missen vooral bij oud kamerleden vanaf 1753. 
Sws filteren van huidige kamerleden handig in de tabel
Bij nieuwe kamerleden staat soms geen geboorteland maar wel geboortestad -> misschien lijst maken met alle nl steden en als stad in steden zitten dan nan=nederland 

wat is gewicht binnen de tabel persoon loopbaan?

huidige kamerleden hebben geen fractielabel?

loopbaan van kamerleden niet gecategoriseerd: zelf doen?
studierichting zelf categoriseren?
werkgever categoriseren?

school mist soms opleidingsnaam maar wel opleidingsinstelling: lijst maken en dan  nieuwe kolom opleidingsniveaus maken? 
ook checken of huidige kamerleden geen missing values hebben met alle relevante kolommen?

persoon id = foreign key in loopbaan




## Dataset begrijpen

In [ ]:
dfbegrijpen= query_df("""SELECT *
                        FROM silver.persoon

""")